# Leave-One-Out Ablation (efficientnetv2 + maxvit)

**런타임: L4 GPU** 필수. 90 run(5 variant × 2 model × 3 fold × 3 seed).

### 세션 한도 대응 — 시드별로 나눠 실행 권장
한 세션에 전부(약 6~9h)는 끊길 수 있음. **셀5의 SEEDS를 한 개씩** 바꿔
3번에 나눠 실행하고, 매번 결과 zip을 다운로드하세요.
- 세션1: SEEDS='1004'  → results 다운로드
- 세션2: SEEDS='2024'  → results 다운로드
- 세션3: SEEDS='777'   → results 다운로드
- 마지막에 로컬에서 3개 합쳐 `python aggregate_loo.py` (CPU·즉시)

In [ ]:
# 1) 클론 + 패키지
%cd /content
!rm -rf fireimage_detection
!git clone https://github.com/yuntaewon812/fireimage_detection.git fireimage_detection -q
!pip install timm grad-cam lime scikit-image scipy -q
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 2) 데이터 업로드 — fireimage_clean.zip (약 32MB) 하나만
#    (학습이므로 가중치 불필요 — 데이터만 있으면 됨)
from google.colab import files
up = files.upload()
print('업로드:', list(up.keys()))

In [ ]:
# 3) 데이터 압축 해제 (normal 792 / abnormal 727 확인)
%cd /content/fireimage_detection
!python colab_setup.py

In [ ]:
# 4) 학습 실행 — 이 세션에서 돌릴 시드만 지정
#    한 시드 = 5 variant × 2 model × 3 fold = 30 model-fold (약 2~3h)
%cd /content/fireimage_detection
import time
SEEDS = '1004'   # ← 세션마다 '1004' / '2024' / '777' 로 바꿔 실행
t0 = time.time()
!python main_ablation_loo.py --seeds {SEEDS} --epochs 15 --patience 5
print(f'\n이 세션 학습 시간: {(time.time()-t0)/60:.1f}분')

In [ ]:
# 5) 이 세션 결과 요약 (부분 — 전체 통계는 3시드 합친 뒤 로컬에서)
%cd /content/fireimage_detection
!python aggregate_loo.py

In [ ]:
# 6) 결과 다운로드 (results/ 의 metrics.csv + xai_sens_stab.csv 만, 작음)
import shutil, glob, os, zipfile
from google.colab import files
OUT = f'/content/loo_results_{SEEDS}.zip'
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob('results/fireimage_loo_*'):
        for f in glob.glob(os.path.join(p, '*.csv')):
            z.write(f, os.path.relpath(f, 'results'))
print('압축:', OUT)
files.download(OUT)

## (선택) 가중치도 보관하려면
세션이 끊겨도 이어학습하려면 weights를 받아두세요. 단 용량 큼.
```python
import shutil
shutil.make_archive(f'/content/loo_weights_{SEEDS}', 'zip', 'model_save')
from google.colab import files; files.download(f'/content/loo_weights_{SEEDS}.zip')
```